# Agent的高级用法-错误处理机制

## 情况1：设置为True/False/固定字符串

举例1：handle_errors 设置为True

In [1]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True  # 默认行为
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='56c30f4f-fae4-4119-a7f6-3d5a1d533692'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 208,
                    'total_tokens': 279,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 32,
                        'engine_ttlt_ms': 264,
                        'pre_inference_ms': 153,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 453,
                        'service_ttlt_ms': 676,
                        'total_duration_ms': 535,
                        'user_visible_ttft_ms': 300
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dn4cg2KWkS3k9PmTWtHEo4Sr7pQiF',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e9353-ebe9-7e40-a037-674bdcaa4c60-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_EwPtOnOpEIqcSBAFLn18IK4b',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_kCreSLpgsCjeezEWOTxo3niq',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 208,
                'output_tokens': 71,
                'total_tokens': 279,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='cac48d94-1dda-44e1-b760-39eb1a35fd03',
            tool_call_id='call_EwPtOnOpEIqcSBAFLn18IK4b'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='2577852e-931f-4a21-a742-a53768c002d3',
            tool_call_id='call_kCreSLpgsCjeezEWOTxo3niq'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 29,
                    'prompt_tokens': 348,
                    'total_tokens': 377,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                 

举例2：handle_errors 设置为False

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=False
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

举例3：handle_errors 设置为固定的字符串

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors="请检查输入的数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='19bcb96c-caf6-4885-8d61-08047f0c154e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 208,
                    'total_tokens': 279,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 39,
                        'engine_ttlt_ms': 258,
                        'pre_inference_ms': 118,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 464,
                        'service_ttlt_ms': 641,
                        'total_duration_ms': 547,
                        'user_visible_ttft_ms': 346
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dn4hXPK2qlE7IkhhUjULq3F5R0gdf',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e9358-8073-7cf1-b3e0-9aaca70bd70c-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_TkNTVYOluuMItVlBesNydocT',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_JtfVNZCc2WW8tQMXtYLEYeMj',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 208,
                'output_tokens': 71,
                'total_tokens': 279,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='请检查输入的数据',
            name='ContactInfo',
            id='779c72c2-7c39-45c3-8285-42cb67f66781',
            tool_call_id='call_TkNTVYOluuMItVlBesNydocT'
        ),
        ToolMessage(
            content='请检查输入的数据',
            name='EventDetails',
            id='0f1c1623-6111-4de0-a959-3f577e9aa0d9',
            tool_call_id='call_JtfVNZCc2WW8tQMXtYLEYeMj'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 303,
                    'total_tokens': 374,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 42,
                        'engine_ttlt_ms': 276,
                        'pre

## 情况2：设置为指定异常类型

举例

In [6]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
        # handle_errors=(StructuredOutputValidationError)
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='0a34cfe1-2784-4588-8852-63e8279eaad4'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 208,
                    'total_tokens': 279,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 6,
                        'engine_ttft_ms': 86,
                        'engine_ttlt_ms': 479,
                        'pre_inference_ms': 87,
                        'service_tbt_ms': 6,
                        'service_ttft_ms': 665,
                        'service_ttlt_ms': 1055,
                        'total_duration_ms': 975,
                        'user_visible_ttft_ms': 578
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dn4mUihR2KKSSlGu8gFqR7Zifkl6I',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e935d-356d-7a91-9677-1c95706fb087-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_mAHfK7EVp6TQaYqmi9O9JESP',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_Wb6l73sgxe8tuiskcSIfDjmU',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 208,
                'output_tokens': 71,
                'total_tokens': 279,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='5facaac5-8b4b-4209-b518-ea4783512ddc',
            tool_call_id='call_mAHfK7EVp6TQaYqmi9O9JESP'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='198cba31-3d24-4c1e-9c03-9fe411b04180',
            tool_call_id='call_Wb6l73sgxe8tuiskcSIfDjmU'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 29,
                    'prompt_tokens': 348,
                    'total_tokens': 377,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                 

## 情况3：设置为自定义错误处理函数

举例

In [7]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from rich import print as rprint

# 自定义错误处理的函数
def custom_error_handler(error : Exception) -> str:
    """自定义错误处理器"""

    error_str = str(error)

    print(f"捕获到的错误类型是：{type(error).__name__}")
    print(f"错误详情:{error_str}")

    if isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回"
    elif isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符号要求。"
    else :
        return f"Error:{error_str}"

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

捕获到的错误类型是：MultipleStructuredOutputsError
错误详情Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='5341ff64-329a-4cb8-a534-203be4585329'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 208,
                    'total_tokens': 279,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 41,
                        'engine_ttlt_ms': 296,
                        'pre_inference_ms': 91,
                        'service_tbt_ms': 4,
                        'service_ttft_ms': 387,
                        'service_ttlt_ms': 635,
                        'total_duration_ms': 552,
                        'user_visible_ttft_ms': 296
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dn4qqqTj6KCRI5Wi62kZYSgrN2Tin',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e9361-507b-7020-93ef-b774e2c3435c-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_Sn9kDCKY7Xy7KF3WH6NTtJx7',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_NifFWtfodixrRuWk5kGSsF9e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 208,
                'output_tokens': 71,
                'total_tokens': 279,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回',
            name='ContactInfo',
            id='43cddf04-3c8c-4a3f-960b-1463692c045e',
            tool_call_id='call_Sn9kDCKY7Xy7KF3WH6NTtJx7'
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回',
            name='EventDetails',
            id='ae69ad99-af2a-4b5b-a5b6-a791b8d863ed',
            tool_call_id='call_NifFWtfodixrRuWk5kGSsF9e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 29,
                    'prompt_tokens': 319,
                    'total_tokens': 348,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 32,
                        'engine_ttlt_ms': 140,
   